# SMART AND — Victimization + Perpetration → Overlap

Este cuaderno calcula el **overlap víctima–perpetrador** mediante una regla **Smart AND**, combinando las predicciones del modelo de victimización y del modelo de perpetración.

La idea es:

```text
pred_overlap = pred_victim AND pred_perpetrator
true_overlap = true_victim AND true_perpetrator
```

**Importante:** las predicciones deben estar alineadas por el mismo identificador o índice original del adolescente. No se deben combinar por posición salvo que se haya comprobado explícitamente que ambos CSV tienen exactamente el mismo orden y los mismos sujetos.


In [1]:
# ============================================================
# 0. Imports y configuración
# ============================================================
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score

# ------------------------------------------------------------------
# RUTAS: cambia estos paths por los CSV reales generados por tus notebooks
# ------------------------------------------------------------------
VICTIM_PRED_PATH = Path("./data/victim_predictions_with_probs.csv")
PERP_PRED_PATH   = Path("./data/perpetrator_predictions_with_probs.csv")

OUTPUT_DIR = Path("./content/smart_and_overlap")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Si no hay una columna de índice/ID común, NO combinar por posición salvo que estés 100% seguro.
ALLOW_POSITIONAL_MERGE = False

# Si tus CSV tienen probabilidades pero no predicción binaria, define umbrales aquí.
# Si ya existe columna y_pred/prediction, se usará esa por defecto.
VICTIM_THRESHOLD = 0.30
PERP_THRESHOLD   = 0.50


## 1. Cargar los CSV de predicciones

El cuaderno espera dos CSV:

- uno de victimización;
- uno de perpetración.

Cada CSV debería contener, idealmente:

- un identificador original del sujeto (`idx_original`, `index`, `ID`, etc.);
- el target real (`y_true`, `true`, `target`, etc.);
- la predicción binaria (`y_pred`, `pred`, `prediction`, etc.) o una probabilidad (`y_prob`, `prob`, `probability`, etc.).


In [2]:
# ============================================================
# 1. Carga y primera inspección
# ============================================================
def read_predictions(path: Path, name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"No existe el fichero de {name}: {path.resolve()}")
    df = pd.read_csv(path)
    print(f"\n{name}: {path}")
    print(f"Shape: {df.shape}")
    print("Columnas:", list(df.columns))
    display(df.head())
    return df

victim_raw = read_predictions(VICTIM_PRED_PATH, "Victimization")
perp_raw = read_predictions(PERP_PRED_PATH, "Perpetration")



Victimization: data/victim_predictions_with_probs.csv
Shape: (942, 4)
Columnas: ['idx_original', 'y_true_victim', 'y_pred_victim', 'y_prob_victim']


,idx_original,y_true_victim,y_pred_victim,y_prob_victim
0,3001,1,0,0.251012
1,1694,0,0,0.251012
2,677,0,0,0.251012
3,840,0,1,0.545689
4,1903,0,1,0.545689



Perpetration: data/perpetrator_predictions_with_probs.csv
Shape: (942, 4)
Columnas: ['idx_original', 'y_true_perp', 'y_pred_perp', 'y_prob_perp']


,idx_original,y_true_perp,y_pred_perp,y_prob_perp
0,1,1,1,0.536061
1,2,0,1,0.560174
2,5,0,1,0.560551
3,11,0,0,0.493554
4,13,0,1,0.545682


## 2. Detección automática de columnas

Si la detección automática falla, edita manualmente las variables `VICTIM_*` y `PERP_*` en la celda siguiente.


In [3]:
# ============================================================
# 2. Detección automática de columnas
# ============================================================
def find_col(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    # búsqueda parcial razonable
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

ID_CANDIDATES = [
    "idx_original", "original_idx", "original_index", "index", "idx", "id", "ID", "Id", "CIP", "subject_id"
]
TRUE_CANDIDATES = [
    "y_true", "true", "target", "label", "y", "real", "actual", "VÍCTIMA", "VICTIMA", "PERPETRADOR"
]
PRED_CANDIDATES = [
    "y_pred", "pred", "prediction", "predicted", "yhat", "class_pred", "pred_label"
]
PROB_CANDIDATES = [
    "y_prob", "prob", "probability", "proba", "score", "risk", "predicted_probability", "prob_1", "p1"
]

VICTIM_ID_COL = find_col(victim_raw, ID_CANDIDATES)
VICTIM_TRUE_COL = find_col(victim_raw, TRUE_CANDIDATES)
VICTIM_PRED_COL = find_col(victim_raw, PRED_CANDIDATES)
VICTIM_PROB_COL = find_col(victim_raw, PROB_CANDIDATES)

PERP_ID_COL = find_col(perp_raw, ID_CANDIDATES)
PERP_TRUE_COL = find_col(perp_raw, TRUE_CANDIDATES)
PERP_PRED_COL = find_col(perp_raw, PRED_CANDIDATES)
PERP_PROB_COL = find_col(perp_raw, PROB_CANDIDATES)

print("Victim columns:", VICTIM_ID_COL, VICTIM_TRUE_COL, VICTIM_PRED_COL, VICTIM_PROB_COL)
print("Perp columns:  ", PERP_ID_COL, PERP_TRUE_COL, PERP_PRED_COL, PERP_PROB_COL)


Victim columns: idx_original y_true_victim y_pred_victim y_prob_victim
Perp columns:   idx_original y_true_perp y_pred_perp y_prob_perp


In [4]:
# ============================================================
# 2b. Sobrescritura manual si hace falta
# ============================================================
# Descomenta y ajusta si la detección automática no ha elegido bien.

# VICTIM_ID_COL = "idx_original"
# VICTIM_TRUE_COL = "y_true"
# VICTIM_PRED_COL = "y_pred"
# VICTIM_PROB_COL = "y_prob"

# PERP_ID_COL = "idx_original"
# PERP_TRUE_COL = "y_true"
# PERP_PRED_COL = "y_pred"
# PERP_PROB_COL = "y_prob"

required = {
    "VICTIM_TRUE_COL": VICTIM_TRUE_COL,
    "PERP_TRUE_COL": PERP_TRUE_COL,
}
missing = [k for k, v in required.items() if v is None]
if missing:
    raise ValueError(f"Faltan columnas obligatorias: {missing}. Ajusta la celda de sobrescritura manual.")

if VICTIM_PRED_COL is None and VICTIM_PROB_COL is None:
    raise ValueError("No encuentro predicción ni probabilidad para victimización.")
if PERP_PRED_COL is None and PERP_PROB_COL is None:
    raise ValueError("No encuentro predicción ni probabilidad para perpetración.")


## 3. Normalizar predicciones

Esta celda construye dos tablas limpias con las columnas:

- `merge_id`
- `y_true_victim` / `y_pred_victim` / `y_prob_victim`
- `y_true_perp` / `y_pred_perp` / `y_prob_perp`


In [5]:
# ============================================================
# 3. Normalización
# ============================================================
def to_binary_series(s):
    """Convierte una serie a 0/1 de forma conservadora."""
    if s.dtype == bool:
        return s.astype(int)
    # strings comunes
    if s.dtype == object:
        mapped = s.astype(str).str.strip().str.lower().map({
            "true": 1, "false": 0,
            "yes": 1, "no": 0,
            "si": 1, "sí": 1,
            "victim": 1, "perpetrator": 1,
            "1": 1, "0": 0,
        })
        if mapped.notna().all():
            return mapped.astype(int)
    return pd.to_numeric(s, errors="raise").astype(int)


def standardize_predictions(df, id_col, true_col, pred_col, prob_col, threshold, prefix):
    out = pd.DataFrame()
    if id_col is not None:
        out["merge_id"] = df[id_col]
    else:
        out["merge_id"] = np.arange(len(df))

    out[f"y_true_{prefix}"] = to_binary_series(df[true_col])

    if prob_col is not None:
        out[f"y_prob_{prefix}"] = pd.to_numeric(df[prob_col], errors="coerce")
    else:
        out[f"y_prob_{prefix}"] = np.nan

    if pred_col is not None:
        out[f"y_pred_{prefix}"] = to_binary_series(df[pred_col])
    else:
        out[f"y_pred_{prefix}"] = (out[f"y_prob_{prefix}"] >= threshold).astype(int)

    return out

victim = standardize_predictions(
    victim_raw, VICTIM_ID_COL, VICTIM_TRUE_COL, VICTIM_PRED_COL, VICTIM_PROB_COL,
    VICTIM_THRESHOLD, "victim"
)

perp = standardize_predictions(
    perp_raw, PERP_ID_COL, PERP_TRUE_COL, PERP_PRED_COL, PERP_PROB_COL,
    PERP_THRESHOLD, "perp"
)

print("Victim standardized:", victim.shape)
print("Perp standardized:", perp.shape)
display(victim.head())
display(perp.head())


Victim standardized: (942, 4)
Perp standardized: (942, 4)


,merge_id,y_true_victim,y_prob_victim,y_pred_victim
0,3001,1,0.251012,0
1,1694,0,0.251012,0
2,677,0,0.251012,0
3,840,0,0.545689,1
4,1903,0,0.545689,1


,merge_id,y_true_perp,y_prob_perp,y_pred_perp
0,1,1,0.536061,1
1,2,0,0.560174,1
2,5,0,0.560551,1
3,11,0,0.493554,0
4,13,0,0.545682,1


## 4. Comprobar alineación y hacer merge

El merge correcto debe hacerse por `merge_id`. Si los dos modelos se han evaluado sobre distintos tests, el número de sujetos comunes puede ser inferior a 942.


In [6]:
# ============================================================
# 4. Comprobación de IDs y merge
# ============================================================
using_real_id = VICTIM_ID_COL is not None and PERP_ID_COL is not None

if not using_real_id and not ALLOW_POSITIONAL_MERGE:
    raise ValueError(
        "No hay columna ID/índice común detectada en ambos CSV. "
        "No combino por posición para evitar un Smart AND inválido. "
        "Añade idx_original a los CSV o pon ALLOW_POSITIONAL_MERGE=True solo si estás 100% seguro."
    )

print("Usando ID real para merge:", using_real_id)
print("Victim IDs únicos:", victim["merge_id"].nunique(), "de", len(victim))
print("Perp IDs únicos:", perp["merge_id"].nunique(), "de", len(perp))

common_ids = set(victim["merge_id"]) & set(perp["merge_id"])
print("IDs comunes:", len(common_ids))
print("Solo victim:", len(set(victim["merge_id"]) - set(perp["merge_id"])))
print("Solo perp:", len(set(perp["merge_id"]) - set(victim["merge_id"])))

merged = victim.merge(perp, on="merge_id", how="inner")
print("Merged shape:", merged.shape)

display(merged.head())


Usando ID real para merge: True
Victim IDs únicos: 942 de 942
Perp IDs únicos: 942 de 942
IDs comunes: 241
Solo victim: 701
Solo perp: 701
Merged shape: (241, 7)


,merge_id,y_true_victim,y_prob_victim,y_pred_victim,y_true_perp,y_prob_perp,y_pred_perp
0,677,0,0.251012,0,0,0.487774,0
1,1903,0,0.545689,1,0,0.595245,1
2,942,1,0.545689,1,0,0.572511,1
3,1579,1,0.545689,1,0,0.512014,1
4,315,0,0.545689,1,0,0.507328,1


## 5. Calcular SMART AND y métricas

Esta es la parte principal:

```text
y_true_overlap = y_true_victim AND y_true_perp
y_pred_overlap = y_pred_victim AND y_pred_perp
```


In [7]:
# ============================================================
# 5. Smart AND
# ============================================================
merged["y_true_overlap"] = (
    (merged["y_true_victim"] == 1) &
    (merged["y_true_perp"] == 1)
).astype(int)

merged["y_pred_overlap"] = (
    (merged["y_pred_victim"] == 1) &
    (merged["y_pred_perp"] == 1)
).astype(int)

print("Distribución y_true_overlap:")
print(merged["y_true_overlap"].value_counts().sort_index())
print("\nDistribución y_pred_overlap:")
print(merged["y_pred_overlap"].value_counts().sort_index())

print("\nClassification report — SMART AND overlap")
print(classification_report(merged["y_true_overlap"], merged["y_pred_overlap"], digits=3))

cm = confusion_matrix(merged["y_true_overlap"], merged["y_pred_overlap"], labels=[0, 1])
TN, FP, FN, TP = cm.ravel()

metrics = {
    "n": len(merged),
    "TP": int(TP),
    "FP": int(FP),
    "TN": int(TN),
    "FN": int(FN),
    "accuracy": accuracy_score(merged["y_true_overlap"], merged["y_pred_overlap"]),
    "balanced_accuracy": balanced_accuracy_score(merged["y_true_overlap"], merged["y_pred_overlap"]),
    "recall_sensitivity": TP / (TP + FN) if (TP + FN) else np.nan,
    "specificity": TN / (TN + FP) if (TN + FP) else np.nan,
    "precision_ppv": TP / (TP + FP) if (TP + FP) else np.nan,
    "npv": TN / (TN + FN) if (TN + FN) else np.nan,
    "f1_positive": 2 * TP / (2 * TP + FP + FN) if (2 * TP + FP + FN) else np.nan,
}

metrics_df = pd.DataFrame([metrics])
display(metrics_df.T.rename(columns={0: "value"}))

print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
print(cm)


Distribución y_true_overlap:
y_true_overlap
0    201
1     40
Name: count, dtype: int64

Distribución y_pred_overlap:
y_pred_overlap
0     78
1    163
Name: count, dtype: int64

Classification report — SMART AND overlap
              precision    recall  f1-score   support

           0      0.974     0.378     0.545       201
           1      0.233     0.950     0.374        40

    accuracy                          0.473       241
   macro avg      0.604     0.664     0.460       241
weighted avg      0.851     0.473     0.517       241



,value
n,241.000000
TP,38.000000
FP,125.000000
TN,76.000000
FN,2.000000
accuracy,0.473029
balanced_accuracy,0.664055
recall_sensitivity,0.950000
specificity,0.378109
precision_ppv,0.233129



Confusion matrix [[TN, FP], [FN, TP]]:
[[ 76 125]
 [  2  38]]


## 6. Guardar resultados

Se guardan:

- `smart_and_predictions.csv`: predicciones individuales y overlap;
- `smart_and_metrics.csv`: métricas principales;
- `smart_and_confusion_matrix.csv`: matriz de confusión.


In [8]:
# ============================================================
# 6. Guardar outputs
# ============================================================
merged.to_csv(OUTPUT_DIR / "smart_and_predictions.csv", index=False)
metrics_df.to_csv(OUTPUT_DIR / "smart_and_metrics.csv", index=False)
pd.DataFrame(cm, index=["true_0", "true_1"], columns=["pred_0", "pred_1"]).to_csv(
    OUTPUT_DIR / "smart_and_confusion_matrix.csv"
)

print("Guardado en:", OUTPUT_DIR.resolve())
print("-", OUTPUT_DIR / "smart_and_predictions.csv")
print("-", OUTPUT_DIR / "smart_and_metrics.csv")
print("-", OUTPUT_DIR / "smart_and_confusion_matrix.csv")


Guardado en: /content/content/smart_and_overlap
- content/smart_and_overlap/smart_and_predictions.csv
- content/smart_and_overlap/smart_and_metrics.csv
- content/smart_and_overlap/smart_and_confusion_matrix.csv


## 7. Comparación rápida con la tabla del manuscrito

La tabla previa del manuscrito para overlap indicaba aproximadamente:

```text
TP = 166
FP = 542
TN = 222
FN = 12
Recall = 93.3%
Precision = 23.4%
Specificity = 29.1%
```

Si este cuaderno devuelve valores muy diferentes, revisar:

1. si los CSV corresponden a los modelos finales correctos;
2. si se está usando el mismo `test_size`;
3. si se está usando el mismo conjunto de sujetos;
4. si los umbrales coinciden;
5. si el merge se ha hecho por índice original y no por posición.
